In [0]:
%pip install xgboost==3.0.0

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.xgboost
import xgboost as xgb

from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from pyspark.sql import functions as F
from sklearn.metrics import r2_score

In [0]:
df = spark.table("cscie103_catalog.final_project.silver_ml_features")
pdf = df.toPandas()

In [0]:
pdf.dtypes

In [0]:
numeric_cols = [
    c for c in pdf.columns 
    if c not in ['date', 'product_family', 'store_type']
]

pdf[numeric_cols] = pdf[numeric_cols].apply(pd.to_numeric, errors='coerce')

pdf = pdf.dropna(subset=['sales'])

In [0]:
pdf["date"] = pd.to_datetime(pdf['date'])

train_data = pdf[pdf["date"] < "2017-01-01"]
validation_data = pdf[(pdf["date"] >= "2017-01-01") & (pdf["date"] < "2017-08-01")]
test_data = pdf[pdf["date"] >= "2017-08-01"]

In [0]:
drop_cols = ['date', 'product_family', 'store_type']

X_train = train_data.drop(columns=drop_cols + ['sales'])
y_train = train_data['sales']

X_valid = validation_data.drop(columns=drop_cols + ['sales'])
y_valid = validation_data['sales']

X_test = test_data.drop(columns=drop_cols + ['sales'])
y_test = test_data['sales']

In [0]:
int_cols = X_train.select_dtypes(include=['int64']).columns
nan_cols = X_train.columns[X_train.isna().mean() == 1.0]

for col in int_cols:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_valid[col] = X_valid[col].fillna(median_val)
    X_test[col]  = X_test[col].fillna(median_val)

X_train = X_train.drop(columns=nan_cols)
X_valid = X_valid.drop(columns=nan_cols)
X_test  = X_test.drop(columns=nan_cols)

In [0]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_valid, label=y_valid)
dtest  = xgb.DMatrix(X_test,  label=y_test)

params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "max_depth": 10,
    "eta": 0.03,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
}

In [0]:
import mlflow
import mlflow.xgboost
from mlflow.models.signature import infer_signature
import xgboost as xgb
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# ============================================================
# MLflow Configuration
# ============================================================
# Set experiment (creates if doesn't exist)
mlflow.set_experiment("/Users/dak6610@g.harvard.edu/retail-sales-forecast")
# Disable autolog - we'll log manually for full control
mlflow.xgboost.autolog(disable=True)
# ============================================================
# Prepare Data - Cast integers to float to avoid schema warnings
# ============================================================
X_train_clean = X_train.astype(np.float64)
X_valid_clean = X_valid.astype(np.float64)
X_test_clean = X_test.astype(np.float64)
# XGBoost DMatrix
dtrain = xgb.DMatrix(X_train_clean, label=y_train)
dvalid = xgb.DMatrix(X_valid_clean, label=y_valid)
dtest = xgb.DMatrix(X_test_clean, label=y_test)
# ============================================================
# Model Parameters
# ============================================================
params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "max_depth": 10,
    "eta": 0.03,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
}
# ============================================================
# Train with MLflow Tracking
# ============================================================
with mlflow.start_run(run_name="xgboost_forecast") as run:
    # Train model
    model = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=5000,
        evals=[(dtrain, "train"), (dvalid, "valid")],
        early_stopping_rounds=50,
        verbose_eval=100
    )
    # ---- Log Parameters ----
    mlflow.log_params(params)
    mlflow.log_param("train_rows", len(X_train))
    mlflow.log_param("valid_rows", len(X_valid))
    mlflow.log_param("test_rows", len(X_test))
    mlflow.log_param("num_features", X_train.shape[1])
    mlflow.log_param("best_iteration", model.best_iteration)
    mlflow.log_param("feature_list", ",".join(X_train.columns.tolist()))
    # ---- Generate Predictions ----
    train_preds = model.predict(dtrain)
    valid_preds = model.predict(dvalid)
    test_preds = model.predict(dtest)
    # ---- Log Metrics ----
    mlflow.log_metric("train_rmse", np.sqrt(mean_squared_error(y_train, train_preds)))
    mlflow.log_metric("valid_rmse", np.sqrt(mean_squared_error(y_valid, valid_preds)))
    mlflow.log_metric("test_rmse", np.sqrt(mean_squared_error(y_test, test_preds)))
    mlflow.log_metric("valid_mae", mean_absolute_error(y_valid, valid_preds))
    mlflow.log_metric("valid_r2", r2_score(y_valid, valid_preds))
    # ---- Create Signature (using float data to avoid warnings) ----
    signature = infer_signature(X_train_clean, y_train)
    # ---- Log Model with Signature ----
    mlflow.xgboost.log_model(
        xgb_model=model,
        artifact_path="model",
        signature=signature,
        input_example=X_train_clean.iloc[:5],
        registered_model_name="store_sales_forecast_model"  # Auto-registers to Model Registry
    )
    run_id = run.info.run_id
    print(f"✓ MLflow Run ID: {run_id}")
    print(f"✓ Best Iteration: {model.best_iteration}")
    print(f"✓ Test RMSE: {np.sqrt(mean_squared_error(y_test, test_preds)):.2f}")

In [0]:
from mlflow import register_model

model_uri = f"runs:/{run_id}/model"

result = mlflow.register_model(
    model_uri=model_uri,
    name="store_sales_forecast_model"
)

In [0]:
train_preds = model.predict(dtrain)
valid_preds = model.predict(dvalid)
test_preds  = model.predict(dtest)

test_preds = np.where(test_preds < 0, 0, test_preds)

In [0]:
print("Best Iteration:", model.best_iteration)
print("Best Validation RMSE:", model.best_score)
print("Train RMSE:", np.sqrt(mean_squared_error(y_train, train_preds)))
print("Valid RMSE:", np.sqrt(mean_squared_error(y_valid, valid_preds)))
print("Test RMSE:",  np.sqrt(mean_squared_error(y_test,  test_preds)))
print("Valid MAE:", mean_absolute_error(y_valid, valid_preds))
print("R^2:", r2_score(y_valid, valid_preds))

-The best RMSE of 243.4 states that the model performs well overall but makes the occasional large mistake. This is heavily penalized by this metric.

-The MAE of 54.5 shows that the predictions are off by about 55 units of sales on average. All errors are treated equally, so large errors are not significantly penalized here.

- The MAE shows that the model performs well on average, but the higher best RMSE shows that the model has some prediction outliers that were significantly off the mark. This is likely due to the holidays, promotions, or other uncommon events.

-The train RMSE is low at 74.2 which is expected since the model was trained with this data. The model fits the training data well

-Validation RMSE is almost the same as the best RMSE which shows consistency and stability

-The test RMSE being 166 is displays a positive signal of the model performing well, and this is almost half of the validation RMSE suggesting that the test is easier to predict. This shows that the model is generalizing well and not overfitting

-The R^2 shows that the model can about 96.81% of the variance in sales, proving that the model is capturing most of the meaningful patterns provided in the feature engineering.


In [0]:
results_df = pd.DataFrame({
    "date": test_data["date"],
    "store_nbr": test_data["store_nbr"],
    "product_family": test_data["product_family"],
    "actual": y_test.values,
    "predicted": test_preds,
    "residual": y_test.values - test_preds
})

spark.createDataFrame(results_df).write.mode("overwrite").saveAsTable(
    "cscie103_catalog.final_project.xgb_predictions"
)